Propósito: Filtra las empresas del SPY500 para traer los papeles en sobreventa. Usa parametros de los indicadores RSI, MACD y Koncorde en la temporalidad de 60min y 7días.

Este filtro busca capturar movimientos que duran entre 2 a 5 días.

In [3]:
%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [12]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [6]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar

print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

Descargando datos para 520 activos...


In [26]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and macd_act >= sig_act 
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


Ticker  RSI actual  RSI anterior    MACD
  AMAT        23.2          24.6 -16.770
  LITE        27.0          27.8 -44.071
  CIEN        27.3          27.6 -15.990
  CDNS        27.4          32.1  -6.046
  NVDA        29.7          32.9  -2.970
  ORCL        31.6          36.8  -1.830
  AKAM        31.8          33.5  -2.972
   GLW        33.8          34.4  -3.879
    BE        33.9          35.1  -8.881
   ETN        34.7          39.4  -7.012


In [16]:
# DESCARGA DATOS DIARIOS
print(f"Descargando datos DIARIOS para {len(tickers_limpios)} activos...")
data_1d = yf.download(tickers_limpios, period="1y", interval="1d", group_by='ticker', progress=False)


Descargando datos DIARIOS para 520 activos...


In [20]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        if (rsi_act <= 45 
            and macd_act >= sig_act 
            ):
        
            empresas_diario.append({
                'Ticker': ticker,
                'Prec_Diario': round(precio_act, 2),
                'RSI_Anterior': round(rsi_ant, 1),
                'RSI_Actual': round(rsi_act, 1),
                'MACD_Diario': round(macd_act, 3),
                'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI_Actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
Ticker  Prec_Diario  RSI_Anterior  RSI_Actual  MACD_Diario  MACD_SIGNAL
   MGM        29.05          21.9        19.9       -2.666       -2.828
  ARES       112.96          34.1        31.9       -3.999       -4.050
    BX       108.23          34.4        32.7       -5.045       -5.144
   MTB       213.85          43.2        32.8       -4.141       -4.596
  AXON       401.39          40.7        33.5      -23.118      -26.532
   PLD       126.84          35.6        35.3       -1.886       -1.917
   RJF       157.08          38.3        37.4       -2.924       -3.253
   CFG        63.11          39.9        37.8       -1.372       -1.478
    GS       890.91          38.2        38.0      -25.402      -28.149
  FSLR       175.46          43.7        39.8       -5.762       -6.967
   PNC       217.66          28.5        40.0       -4.825       -5.344
  JBHT       226.15          52.4        40.4       -6.233       -7.930
   DOC        19.02          38.9

In [21]:
# DESCARGA DATOS SEMANAL.
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

Descargando datos SEMANALES para 520 activos...


In [25]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 40 and macd_act <= sig_act:
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI_Act': round(rsi_act, 1),
                'RSI_Ant': round(rsi_ant, 1),
                'MACD_S': round(macd_act, 3),
                'MACD_S_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI_Act').to_string(index=False))
else:
    print("Sin señales semanales.")


PAPELES DETECTADOS EN SEMANAL.
Ticker  RSI_Act  RSI_Ant   MACD_S  MACD_S_SIGNAL
   MGM     17.0     17.2   -2.336         -0.216
  WYNN     24.7     21.3   -7.856         -6.046
   PNR     24.8     20.6   -8.627         -8.470
   URA     26.9     33.1   -2.367         -1.746
  SKYD     27.4     40.5   -0.578         -0.526
   ALB     28.4     30.2  -12.936         -9.205
    GS     28.9     29.6   -5.695         17.729
   LII     28.9     27.8  -40.371        -30.849
    MS     28.9     34.1    0.498          5.471
   APP     29.7     30.4  -61.166        -55.814
   PLD     29.7     33.7   -1.528          0.508
   HWM     30.2     31.4   -4.701          2.876
   CRH     30.3     23.9   -7.429         -6.300
   FRT     30.4     30.3   -0.590          1.645
   PCG     30.7     30.8   -1.000         -0.532
   BAC     32.1     37.4    0.292          1.525
    BX     32.4     36.8   -3.198         -1.120
   ROL     32.5     24.5   -5.408         -5.340
    BA     32.6     33.2   -7.100    

In [ ]:
# ALINEAR TEMPORALIDAD 60MIN Y 1D.

df_60 = pd.DataFrame(empresas_60min)
df_sem = pd.DataFrame(empresas_semanal)
print("Alineando las temporalidades.")
if not df_60.empty and not df_sem.empty:
    activos = pd.merge(df_60, df_sem, on='Ticker')


    if not activos.empty:
        print("\n" + "!"*30)
        print("ALINEACIÓN DE TEMPORALIDAD DETECTADA")
        print("Activos en zona de pánico tanto en 60min como en 7d:")
        # Seleccionamos las columnas de la comparativa.
        columnas_finales = [
            'Ticker', 'Prec_Act', 'RSI_Act',
            'RSI_Sem'
        ]
        print(activos[columnas_finales].to_string(index=False))

        print("\nInterpretación: El activo está en suelo macro y dando señal de giro micro.")
    else:
        print("\nNo hay coincidencias 'AND'.")
        print("Existen gangas individuales, pero no están alineadas en ambos fractales hoy.")
else:
    if df_60.empty:
        print("- El fractal de 60 min no detectó señales.")
    if df_sem.empty:
        print("- El fractal Semanal no detectó señales.")
    print("\nNo se puede realizar la operación AND sin datos en ambos marcos.")

Alineando las temporalidades.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ALINEACIÓN DE TEMPORALIDAD DETECTADA
Activos en zona de pánico tanto en 60min como en 7d:


KeyError: "['Konk_Act', 'Konk_S_Act'] not in index"

In [ ]:
# SEMÁFORO Y REVERSIÓN DE CALIDAD

UMBRAL_PANICO = 35       # Para activos "estirados" al extremo (Cuchillos)
UMBRAL_CALIDAD = 48      # Para captar reversiones inminentes en líderes
EMPRESAS_FILTRADAS = []

print(f"Iniciando escáner sobre {len(tickers_limpios)} activos...")

for ticker in tickers_limpios:
    try:
        # Reutilizamos los datos de la celda de 60 min (Micro Fractal)
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos (Referenciando el fractal de 1h)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)

        # Variables de control
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        # FILTRO ESTIRADO (Filtro de Entrada)
        # Captura el pánico real O sobreventa relativa en empresas fuertes
        if rsi_act < UMBRAL_CALIDAD and kon_act < 0:

            # LÓGICA DEL SEMÁFORO (El Gatillo)
            # Caso Verde: Cruce de Media al alza (Ayer < MA y Hoy > MA)
            if rsi_act > rsi_ma and rsi_ant <= rsi_ma:
                estado = "🟢 GATILLO (Cruce confirmado)"
            # Caso Amarillo: Frenado de caída (Ayer < Hoy pero aún debajo de MA)
            elif rsi_act > rsi_ant:
                estado = "🟡 AGOTAMIENTO (En observación)"
            # Caso Rojo: Inercia bajista (Sigue cayendo)
            else:
                estado = "🔴 CUCHILLO (Sin señal de giro)"

            # Buscamos si el ticker también dio señal en la lista semanal previa
            es_ganga_macro = any(e['Ticker'] == ticker for e in empresas_semanal)
            etiqueta_macro = "🔥 ALINEADO (Macro+Micro)" if es_ganga_macro else "Solo Micro"

            EMPRESAS_FILTRADAS.append({
                'Ticker': ticker,
                'Estado': estado,
                'RSI': round(rsi_act, 1),
                'RSI_MA': round(rsi_ma, 1),
                'Tipo': "Extremo" if rsi_act < UMBRAL_PANICO else "Calidad",
                'Fractal': etiqueta_macro
            })

    except Exception:
        continue

# PRESENTACIÓN DE RESULTADOS
print("\n" + "="*100)
print(f"REPORTE DEL S&P500 - {len(EMPRESAS_FILTRADAS)} ACTIVOS EN RADAR")
print("="*100)

if EMPRESAS_FILTRADAS:
    df_final = pd.DataFrame(EMPRESAS_FILTRADAS)

    # Ordenamos por prioridad: Gatillos primero, luego Agotamiento, luego Cuchillos
    orden_estado = {"🟢 GATILLO (Cruce confirmado)": 0, "🟡 AGOTAMIENTO (En observación)": 1, "🔴 CUCHILLO CAYENDO (Sin señal de giro)": 2}
    df_final['Sort'] = df_final['Estado'].map(orden_estado)
    df_final = df_final.sort_values(by=['Sort', 'RSI']).drop(columns='Sort')

    print(df_final.to_string(index=False))
else:
    print("El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.")

Iniciando escáner sobre 520 activos...

REPORTE DEL S&P500 - 0 ACTIVOS EN RADAR
El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.
